In [ ]:
# Quantum simulation libraries
from qutip import (
    basis, 
    mesolve, 
    qeye, 
    sigmax, 
    sigmay, 
    sigmaz, 
    tensor,
    )
import qutip

# Machine learning libraries
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

# Plotting libraries
import matplotlib.pyplot as plt
from rich.progress import track
import seaborn as sns

# Linalg libraries
import numpy as np
from scipy.stats import pearsonr

# Helper libraries
from dataclasses import dataclass


In [ ]:
# set the device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Load target data

In [ ]:
class BFieldGenerator:
    def __init__(
        self,  amplitude: float, frequency: float, resolution: int = 0.01, length: int = 1000
        ):
        """
        Create the BField generator.

        Parameters
        ----------
        amplitude : float
            Amplitude of the magnetic field.
        frequency : float
            Frequency of the cosine wave.
        resoltion : float
            Distance between measured points.
        length : int
            Amount of time to run it for.
        """
        self.amplitude = amplitude
        self.frequency = frequency

        self.counter = 0

        self.measured_field = []
        self.times = []

    def __call__(self, t: float):
        """
        Get the next value of the magnetic field.

        Parameters
        ----------
        t : float
            Current time, but this is ignored.
        """
        b_field = self.amplitude * np.cos(self.frequency * t)
        self.measured_field.append(b_field)
        self.times.append(t)
        self.counter += 1

        return b_field

# Run Simulation

In [ ]:
@dataclass
class SimulationParameters:
    """
    Helper class for the simulation parameters.
    """
    length: int
    coupling: list

@dataclass
class SimulationState:
    """ 
    Helper class for the simulation state.
    """
    number_of_spins: int
    quantum_state: list
    spin_list: list
    coupling_list: list

In [ ]:
def get_simulation_state(parameters: SimulationParameters):
    """
    Returns the initial state of the simulation.
    """
    # Get the initial wavefunction
    number_of_spins = parameters.length

    initial_state = [basis(2, 1)] + [basis(2, 0)] * (number_of_spins - 1)

    # Setup operators for individual qubits
    sx_list, sy_list, sz_list = [], [], []
    for i in range(number_of_spins):
        op_list = [qeye(2)] * number_of_spins
        op_list[i] = sigmax()
        sx_list.append(tensor(op_list))
        op_list[i] = sigmay()
        sy_list.append(tensor(op_list))
        op_list[i] = sigmaz()
        sz_list.append(tensor(op_list))

    # Setup the operators for the coupling
    Jx = parameters.coupling * np.ones(number_of_spins)
    Jy = parameters.coupling * np.ones(number_of_spins)
    Jz = parameters.coupling * np.ones(number_of_spins)    

    return SimulationState(
        number_of_spins=number_of_spins,
        quantum_state=tensor(initial_state),
        spin_list=[sx_list, sy_list, sz_list],
        coupling_list=[Jx, Jy, Jz],
    )

In [ ]:
def compute_hamiltonian(t, args):
    """
    Compute the Hamiltonian at time t.
    
    Parameters
    ----------
    t : float
        Current time.
    args : dict
        System parameters in the H computation.
    """
    sx_list = args["sx_list"]
    sy_list = args["sy_list"]
    sz_list = args["sz_list"]
    Jx = args["Jx"]
    Jy = args["Jy"]
    Jz = args["Jz"]
    driving_field = args["driving"]

    N = args["N"]

    # Magnetic field terms to top row
    H = 0
    for i in range(5):        
        H -= driving_field(t) * sz_list[i]
        H -= driving_field(t) * sx_list[i]
        H -= driving_field(t) * sy_list[i]

    
    # Interaction terms
    for n in range(N - 1):
        H += -0.5 * Jx[n] * sx_list[n] * sx_list[n + 1]
        H += -0.5 * Jy[n] * sy_list[n] * sy_list[n + 1]
        H += -0.5 * Jz[n] * sz_list[n] * sz_list[n + 1]

    return H

In [ ]:
# for strength in np.linspace(1, 10, 10, dtype=int):
simulation_parameters = SimulationParameters(
    length=5,
    coupling=10. * np.pi,
)
simulation_state = get_simulation_state(simulation_parameters)

field_generator = BFieldGenerator(1.0, 0.1 * np.pi, length=20)
args = {
    "sx_list": simulation_state.spin_list[0],
    "sy_list": simulation_state.spin_list[1],
    "sz_list": simulation_state.spin_list[2],
    "Jx": simulation_state.coupling_list[0],
    "Jy": simulation_state.coupling_list[1],
    "Jz": simulation_state.coupling_list[2],
    "N": simulation_state.number_of_spins,
    "driving": field_generator
}

In [ ]:
times = np.linspace(0, 10, 100)


In [ ]:
data = []

for t in times:
    data.append(field_generator(t))

In [ ]:
plt.plot(data)

In [ ]:
times = np.linspace(0, 10, 100)
results = mesolve(compute_hamiltonian, simulation_state.quantum_state, times, [], [], args)

# qutip.qsave(states, f"{strength}")

In [ ]:
fit_field = BFieldGenerator(1.0, 0.1 * np.pi, length=20)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 10))

for item in results.states:
    density_matrix = np.concatenate(np.array(item * item.dag()))
    sns.kdeplot(np.real(density_matrix), ax=ax[0])
    sns.kdeplot(np.imag(density_matrix), ax=ax[1])

ax[0].set_ylim(0, 1.0)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 10))

for item in results.states:
    density_matrix = np.concatenate(np.array(item * item.dag()))
    sns.kdeplot(np.real(density_matrix), ax=ax[0])
    sns.kdeplot(np.imag(density_matrix), ax=ax[1])

ax[0].set_ylim(0, 1.0)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 10))

for item in results.states:
    density_matrix = np.concatenate(np.array(item * item.dag()))
    sns.kdeplot(np.real(density_matrix), ax=ax[0])
    sns.kdeplot(np.imag(density_matrix), ax=ax[1])

ax[0].set_ylim(0, 1.0)
plt.show()